# `C07` — Immediate-or-cancel

Scenario **C07** of the Track A certification ladder (card `OMIE-CERT-C07-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C7). Oracle kind: **exact-parametric, carried by M3**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** on order ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW) and the engine's trade-row timestamps, or a structural equality of lists of them. Euro figures are shown, never asserted; the one figure read back from a float (the engine's reported trade price) is converted to ticks and held to a one-tick bracket, labelled `ENGINEERING INVARIANT`.

## 1. Purpose

**Purpose.** The first scenario carried primarily by a **metamorphic** relation rather than a computed
answer, included partly to demonstrate the technique on something simple. IOC and NON are the *same
order* as far as matching is concerned; they diverge only afterwards, in what happens to the unfilled
remainder. So the trade lists must be identical (M3), and nobody needs to know what either contains
to check that. Three regimes — no liquidity, partial, full — are drawn every time, because the
interesting IOC bug only appears when there is a leftover to dispose of.

*(Verbatim from `scenario-ladder.md` §C7.)*

**Certifies** (card): §4.5 table, IOC row; §6.2 item 5; §5.1's branch `CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; halt/hibernation)`, quoted whole. **Pass** (card): M3 equality plus a zero-residual assertion.

**How the claim is pinned.** Every draw of `C07.ioc_regimes` is two orders on one contract, on a fresh book: one resting NON **maker** (submission 0), then one **IOC aggressor** on the other side (submission 1). The draw's own *counterfactual* tape (`draw.aux_tapes["counterfactual"]`) is the same two rows, byte for byte, except that the aggressor is NON. Both tapes are run. The generator draws one of three regimes per seed:

- **none** — the aggressor's limit does not cross the maker's (one euro short): no trade; the IOC is cancelled whole, the NON rests whole.
- **partial** — the limit crosses and the aggressor is larger than the maker: one trade for the maker's whole size; the IOC's leftover is cancelled, the NON's leftover rests.
- **full** — the limit crosses and the aggressor is at most the maker's size: one trade for the aggressor's whole size; nothing is left over in either run, and the maker keeps the rest of its size.

Per draw, separately (section 5): **(Q)** the IOC run's exact counterparty sequence; **(M3)** the IOC run's trade list equal, row for row and **field for field**, to the NON run's; **(R)** the zero-residual assertion — the IOC's own order id rests in no book after its submission, and its quantity splits exactly into filled and cancelled; **(B)** both final books, and the M3 book relation (the IOC book is the NON book less the NON aggressor's own resting row); **(A)** every order accepted in both runs; **(E)** the engine's own report of each submission; and the battery: RB and I1–I9 on both runs, plus the battery's own M3 check.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card cites `market-spec.md` v2.12; the file is now v2.13. v2.13 changed §4.2.1 Rule 3 and §6.2 item 7 only (OQ-31). Neither is quoted here, and §6.2 items 1–5 are unchanged. The passages in section 2 are checked against the current file at run time.
- **F2 — the order status is not observable.** The engine reports, per submission, only `accepted`, `traded`, a trade quantity, a trade price and a settlement value (`EngineReport`); it stores no per-order status. So the card's "status `CANCELLED` when partial" cannot be read off the engine. It is certified through its **observable signature** instead (R): the IOC's order id is absent from every book after its submission (read from engine snapshots), and its quantity splits as `filled` (read from the engine's trade rows) plus a remainder that is neither resting, hidden nor evicted (read from the final book). The record's `ioc_remainder` bucket is **computed by the record builder from order semantics** (accepted quantity minus the IOC's own fills), not read from the engine, so on its own it would prove nothing; it is recorded beside the engine-read buckets, and it is the engine-read ones that carry the assertion. The status label of each regime (`CANCELLED` for none and partial, `EXECUTED` for full) is stated in section 5 and in the results, not asserted as an engine value.
- **F3 — §5.1 has no first-level branch for a partially executed IOC.** Under `ACCEPTED` the diagram lists `EXECUTED`, `PARTIALLY_EXECUTED + resting in BOOK`, `CANCELLED (IOC/FOK unmatched; …)` and `IN_BOOK`. A partially executed IOC is neither unmatched nor resting. The only branch naming it, `PARTIALLY_EXECUTED (residual remains in book or cancelled per IOC)`, sits under `IN_BOOK`, which an IOC never enters (Matching Algorithm §3.5.2: "An IOC order is never displayed in the order book"). This notebook follows the card: a partial IOC is `PARTIALLY_EXECUTED`, then its remainder is `CANCELLED` (§6.2 item 5, §4.5 IOC row). Because the status is unobserved (F2), the reading changes no assertion. It is recorded for the `domain-analyst`; no open question is filed by K2.
- **F4 — "three regimes drawn every time" is met by coverage, not per seed.** The registered construction draws the regime from each seed's stream, so one seed is one regime. Every run covers all three regimes on both resting sides through a **coverage assertion** (section 7): each of the six classes regime × resting side must appear among the valid draws (the default seeds 0–499 hold at least 70 of each), and each class must have its eager comparison. The explicit-regime form `make_ioc_regimes(seed, regime=…)` is not used, so that one seed names one draw in the results and in a frozen counterexample.
- **F5 — no competition at one price; M3 is compared on every field.** Each draw has one maker, so there is no price-time tie and the counterparty sequence (Q) is one row at most; it is still pinned exactly (domain review §16–17, D3-08 practice). The battery's M3 compares a six-field key per trade (price, quantity, both order ids, both agents). This notebook's (M3) assertion is stricter: **every** `Fill` field, including the trade row index, the submission, both sides' order ids and agents, and the trade's timestamp (`sec`, `nsec`). §5.1 and §6.2 make IOC and NON identical until the remainder is disposed of, which happens after the trade is written; so nothing in the trade may differ.
- **F6 — which regime carries which half.** In **none** both trade lists are empty, so (M3) on trades is vacuous by construction and the regime is carried by (R): an IOC that finds no liquidity is still **accepted** and then cancelled whole, never rejected and never rested. In **full** nothing is left over, so IOC and NON must agree on the book as well, and the maker keeps $q_{\text{rest}} - q_{\text{agg}}$ ("the resting maker legitimately survives", M3). **partial** is the regime where both halves bite: the trade lists must agree while the books must differ by exactly the NON's resting leftover.
- **F7 — the engine's reported trade price.** The spec does not define a per-submission reported price. With one trade per submission any reading (average, first or last) is the maker's limit, so on a traded submission the reported price, converted to ticks, must lie within one tick of $p_{\text{rest}}$ (the slack absorbs the float32 decode). This is an `ENGINEERING INVARIANT`. On an untraded submission the reported price is not asserted.

## 2. The rule, quoted

The governing passages, quoted **verbatim** and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **What IOC means (§1, §4.5).** An IOC may fill partially, and what it does not fill never enters the book: "No (cancelled)". A NON may fill partially, and what it does not fill enters the book.
- **Matching is the same for both (§6.1, §6.2 items 1–3).** The incoming order meets the best opposite order while `sell_price ≤ buy_price`, and each trade prints at the resting order's limit. None of these items mentions the execution restriction: IOC and NON trade identically (M3).
- **The leftover (§6.2 items 4 and 5), the rule certified.** A partially filled NON's residual enters the book; a partially filled IOC's residual is cancelled.
- **The status (§5.1).** An unmatched IOC is `CANCELLED (IOC/FOK unmatched; …)`, the branch the card certifies, quoted whole; a fully matched order is `EXECUTED`. The partially executed IOC is discussed in finding F3.
- **Why the zero-residual check can fail (domain review §16.1).** The battery's I2 counts an IOC remainder as cancelled from the IOC's own fills only, so an engine that rests it counts it twice.
- **The regulatory text** (citations): the Matching Algorithm's §3.5.1–§3.5.2 define NON and IOC in the same words up to the leftover, and §5.1.3–§5.1.4 say when the incoming order is deleted rather than entered.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §1 glossary and §4.5: what the IOC restriction means ------------------------------------------
    SpecQuote(
        "market-spec.md", "§1",
        "| **IOC** | Immediate or Cancel — fill what is available immediately, cancel remainder. |",
        note="The glossary's definition: the IOC trades what it can at once, and its remainder is cancelled.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| Restriction | Partial fill | Book entry if unmatched | Applicable to |",
        note="Header of the execution-restriction table, quoted so the two rows below can be read.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| IOC | Yes | No (cancelled) | Regular |",
        note="The row certified: partial fill permitted, book entry not. The IOC's own id never rests (R).",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| NON | Yes | Yes | Regular, Block |",
        note="The counterfactual: the same order as NON fills the same way, and its unmatched part rests.",
    ),
    # --- §5.1: the state model, the cancellation branch quoted whole ------------------------------------
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; halt/hibernation)",
        note="The branch certified, quoted whole: an IOC that is not matched is cancelled (regime none); "
             "a partially matched IOC's remainder is cancelled (regime partial; finding F3).",
    ),
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ EXECUTED (fully matched immediately)",
        note="Regime full: the IOC is matched in full on arrival, with nothing left to cancel.",
    ),
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ PARTIALLY_EXECUTED (residual remains in book or cancelled per IOC)",
        note="The only branch naming a partially executed IOC; it sits under IN_BOOK (finding F3).",
    ),
    # --- §6.1 / §6.2: matching does not depend on the restriction; the leftover does --------------------
    SpecQuote(
        "market-spec.md", "§6.1",
        "A match occurs when best bid price ≥ best ask price.",
        note="Regime none: the aggressor's limit is one euro short of the maker's, so no match occurs.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 1",
        "Incoming order attempts to match against best counterpart in the consolidated order book.",
        note="The aggressor is the incoming order; the maker is the only counterpart.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 2",
        "Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.",
        note="The crossing test of the reference walk (W). It does not mention the execution restriction.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="The one trade prints at the maker's limit, in both runs.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 4",
        "If incoming order is partially filled and has NON restriction: residual enters the book with its "
        "original entry timestamp (see §5.2).",
        note="The counterfactual's leftover rests: the NON book differs from the IOC book by exactly this row.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 5",
        "If incoming order is partially filled and has IOC restriction: residual is cancelled.",
        note="The rule certified: the IOC's leftover is cancelled, never rested.",
    ),
    # --- the battery's I2 and the zero-residual check ----------------------------------------------------
    SpecQuote(
        "scenario-ladder-review.md", "§16.1",
        "`ioc_remainder` subtracts only the IOC's own aggressor fills, never resting quantity. An engine that\n"
        "  wrongly rests an IOC remainder therefore counts it twice and fails I2.",
        note="Why I2 can catch a rested IOC remainder, beside this notebook's own zero-residual check (R).",
    ),
)
CITATIONS = (
    Citation(
        "Continuous Trading Matching Algorithm (public description) §3.5.1 — "
        "docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_"
        "Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md",
        "An order submitted with the execution restriction NON is either executed im mediately, or, if the order "
        "cannot be matched right away, entered into the order book. Partial order executions are allowed and NON "
        "orders can be executed against multi ple other orders and create multiple trades.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §3.5.2 — same file",
        "An order submitted with the execution restriction IOC is either executed immediately , or, if the order "
        "cannot be matched, deleted without entry in the order book. Partial executions are allowed and IOC "
        "orders can be executed against multiple other orders and create multiple trades. An IOC order is never "
        "displayed in the order book.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1.3 — same file",
        "This process continuous as long as the incoming order remains executable and has a positive order "
        "quantity. Subsequently , the order is deleted if the order quantity has reached zero or if it has the "
        "IOC execution restriction. In all other cases, the order is entered into the order book with its "
        "remaining quantity.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1.4 — same file",
        "If an order with a new timestamp cannot be executed against any existing order, it is entered into the "
        "order book, unless it has the execution instruction IOC or FOK. If it has the execution restriction IOC "
        "or FOK, it is deleted.",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws of `C07.ioc_regimes` (the card's default for a random scenario). Each valid draw runs **twice**, the IOC tape and its NON counterfactual, both jitted.
- **`SEED0 = 0`** is the first seed. A draw found outside the capacity margin before the run (either tape), or marked I8 `invalid` by the battery after it (either run), is **discarded and replaced by the next seed**. The loop stops at `DRAWS` valid draws or after `2 × DRAWS` attempts, and the discard rate is reported.
- **`EAGER_DRAWS = 3`.** The first `EAGER_DRAWS` **valid** draws also run eager, both tapes, and each eager record is compared, integer field by integer field, with its jitted record (`compare_records`). On top of these, an eager run of both tapes is **forced** for the first valid draw of each of the six classes regime × resting side. Every one of these twelve class comparisons (six classes × two tapes) **must** exist: a missing one is recorded as a failed `eager_jit` assertion. Eager runs are chosen among valid draws, so a discarded seed cannot remove them.
- The tick grids are **not** parameters: they come from `OMIEProductConfig()` and serve only to display euros and to convert the engine's float report to ticks in (E).

A run with far fewer draws than the default can miss a class; it then fails the coverage check and that class's eager comparisons. That is intended: such a run does not cover the card's three regimes.

In [ ]:
CERT_ID = "C07"     # C00-C16 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500         # valid draws of C07.ioc_regimes (card: random scenarios default to 500 draws)
SEED0 = 0           # first seed
EAGER_DRAWS = 3     # the first EAGER_DRAWS valid draws also run eager (plus the first of every regime x side class)

In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

CONSTRUCTION = "C07.ioc_regimes"
REGIMES = ("none", "partial", "full")
MAX_ATTEMPTS = 2 * DRAWS   # the draw loop stops at DRAWS valid draws or after 2 x DRAWS attempts

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
EXEC_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS = int(Validity.GFS)

SLICE_MARGIN = 64                     # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800             # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C07_ioc.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION} x {DRAWS} valid draws from seed {SEED0} (at most {MAX_ATTEMPTS} attempts); "
      f"each draw runs its IOC tape and its NON counterfactual")
print(f"eager: first {EAGER_DRAWS} valid draws + the first of each regime x resting-side class, both tapes")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display and report conversion only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")

## 4. Dataset

**`C07.ioc_regimes`** (`omie_tape_synth.make_ioc_regimes`, regime drawn from the seed's stream) emits two GFS rows on one contract, order ids 1 and 2 and tape timestamps 1 and 2 in row order. Write $\sigma$ for the maker's side and $\bar\sigma$ for the other.

| row (submission $s$) | id | side | limit | size | restriction: IOC tape / counterfactual |
|---|---|---|---|---|---|
| maker, $s = 0$ | $o_r = 1$ | $\sigma$ | $p_r$, 30–90 EUR/MWh | $q_r$, 5–30 MW | NON / NON |
| aggressor, $s = 1$ | $o_a = 2$ | $\bar\sigma$ | $p_a = p_r \mp 100$ ticks (none: one euro short of crossing) or $p_r \pm 100$ ticks (partial, full: one euro through) | none: 1–10 MW; partial: $q_a > q_r$; full: $q_a \le q_r$ | **IOC** / **NON** |

**What section 5 reads.** The expectation reads the **tape rows** of both tapes through one reference walk, and `draw.params` for the closed form, which is cross-checked against the walk on every draw and checked against the tape. The regime is re-derived from the tape (does $p_a$ cross $p_r$; is $q_a > q_r$) and must equal `params["regime"]`. `draw.expected` is never read.

**The precondition, checked from the tapes for every valid draw** (an aggregate assertion in section 7):

- **Shape.** Two real rows, one contract, order ids $[1, 2]$ equal to `params`' `rest_oid`, `agg_oid`; tape timestamps strictly increasing; prices and sizes within §2.3's bounds; both GFS, no peak, no price increment.
- **Restrictions.** The maker is NON; the aggressor is **IOC** on the tape and **NON** on the counterfactual; the counterfactual equals the tape in every other cell, pad rows included (M3 compares the same price and quantity).
- **Sides and values.** Maker on `rest_side`, aggressor on `agg_side`, the opposite side. `p_rest_int`, `q_rest_int`, `p_agg_int`, `q_agg_int` equal the tape's values.
- **The regime.** The regime re-derived from the tape equals `params["regime"]`.
- **No icebergs.** None (so $\lceil qty/peak\rceil$ is 0).

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings). Two orders sit far inside it: at most one resting per side, at most two trade units, a hop bound of four. It is still checked **before** each run, on both tapes: a tape whose analytic bounds exceed the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs), or whose largest $\lceil qty/peak \rceil$ exceeds 64, is never run; the draw is discarded and re-drawn. After the run, the battery's I8 confirms the flags on both records.

Below: the first seed of each regime in readable units beside the integers, the book shapes, and the class counts of the initial seed range.

In [ ]:
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_PEAK, C_PPD = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC,
                                                    synth.COL_VALIDITY, synth.COL_PEAK, synth.COL_PPD)


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def crosses(rest_side, p_rest, p_in):
    # §6.1 / §6.2 item 2: sell price <= buy price, for a resting order on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def max_slices(tape):
    # Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none).
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((-(-int(r[C_QTY]) // max(1, int(r[C_PEAK]))) for r in ice), default=0)


def counterfactual(draw):
    # The NON counterfactual tape of a draw (None when the draw has none).
    cf = draw.aux_tapes.get("counterfactual") if draw.aux_tapes else None
    return None if cf is None else np.asarray(cf)


DRAWN = {}


def prepare(seed):
    # Draw one seed (the run loop of section 7 calls this for a re-draw too).
    d = synth.CONSTRUCTIONS[CONSTRUCTION](seed)
    DRAWN[seed] = d
    return d


for _s in range(SEED0, SEED0 + DRAWS):
    prepare(_s)


def tape_table(draw, title):
    cf = counterfactual(draw)
    cf_exec = {int(r[C_OID]): int(r[C_EXEC]) for r in real_rows(cf)} if cf is not None else {}
    rows = ["| s | role | oid | side | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | tape ts | "
            "exec (IOC tape) | exec (counterfactual) |",
            "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(draw.tape)):
        role = "maker" if k == 0 else "aggressor"
        oid = int(r[C_OID])
        rows.append(f"| {k} | {role} | {oid} | {side_name(r[C_SIDE])} | "
                    f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_AGENT])} | {int(r[C_TS])} | "
                    f"{EXEC_NAME.get(int(r[C_EXEC]), int(r[C_EXEC]))} | "
                    f"{EXEC_NAME.get(cf_exec.get(oid), cf_exec.get(oid))} |")
    return f"**{title}**\n\n" + "\n".join(rows)


FIRST_OF = {}
for _s in sorted(DRAWN):
    FIRST_OF.setdefault(DRAWN[_s].params.get("regime"), _s)
for _r in REGIMES:
    if _r not in FIRST_OF:
        print(f"regime {_r}: no draw in the initial seed range")
        continue
    _d = DRAWN[FIRST_OF[_r]]
    display(Markdown(tape_table(_d, f"regime {_r}: seed {FIRST_OF[_r]} (digest `{_d.digest()[:16]}…`)")))
    print(f"seed {FIRST_OF[_r]} params: {_d.params}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, _r in zip(axes, REGIMES):
    if _r not in FIRST_OF:
        ax.set_title(f"regime {_r}: no draw")
        continue
    _d = DRAWN[FIRST_OF[_r]]
    _m, _a = real_rows(_d.tape)[:2]
    for row, lbl, col in ((_m, "maker (NON)", None), (_a, "IOC aggressor", None)):
        buy = int(row[C_SIDE]) == BUY
        q = decode_qty(int(row[C_QTY]), CONFIG)
        p = decode_price(int(row[C_PRICE]), CONFIG)
        ax.barh(p, q * (1 if buy else -1), height=0.4, color="tab:green" if buy else "tab:red",
                alpha=1.0 if row is _m else 0.45, hatch=None if row is _m else "//",
                label=f"{lbl}: {side_name(row[C_SIDE])} {q:.1f} MW @ {p:.2f}")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("quantity (MW): buy right, sell left")
    ax.set_ylabel("limit (EUR/MWh)")
    ax.set_title(f"regime {_r}, seed {FIRST_OF[_r]}", fontsize=9)
    ax.legend(loc="best", fontsize=7)
plt.tight_layout()
plt.show()

_classes = Counter((d.params.get("regime"), side_name(d.params.get("rest_side", BUY))) for d in DRAWN.values())
display(Markdown("**Classes in the initial seed range** (regime x resting side, counts)\n\n"
                 "| regime | resting side | draws |\n|---|---|---|\n"
                 + "\n".join(f"| {r} | {s} | {_classes.get((r, s), 0)} |" for r in REGIMES for s in ("BUY", "SELL"))))

## 5. Known result

**Notation.** Integer ticks throughout. The maker is $(o_r, \sigma, p_r, q_r)$ at submission 0, the aggressor $(o_a, \bar\sigma, p_a, q_a)$ at submission 1; both tapes are identical except the aggressor's restriction (section 4).

**(W) The reference walk** (§6.1, §6.2 items 1–5, §4.5). Process the rows in submission order. An entering order with limit $p$ and size $q$ meets the opposite side ranked by $(p_e, s_e)$ (SELL resting: lowest price first) or $(-p_e, s_e)$ (BUY resting: highest price first). While it has quantity left and the best opposite entry crosses (§6.2 item 2), it trades $f = \min(\text{left}, q_e)$ **at $p_e$** (§6.2 item 3), and an entry used up leaves the book. **The restriction plays no part in the loop.** Then, and only then, the leftover is disposed of: a NON leftover rests at its own limit (§6.2 item 4, §4.5 NON row); an IOC leftover is cancelled (§6.2 item 5, §4.5 IOC row; §5.1 `CANCELLED (IOC/FOK unmatched; …)` when nothing traded).

**Closed form** (from `draw.params`). Let $f$ be the traded quantity of the regime:

$$f = \begin{cases} 0 & \text{none} \\ q_r & \text{partial } (q_a > q_r) \\ q_a & \text{full } (q_a \le q_r) \end{cases}$$

$$\text{fills}_{\text{IOC}} = \text{fills}_{\text{NON}} = \big[\,[o_r,\ o_a,\ p_r,\ f,\ 1]\ \text{if } f > 0\,\big]$$

$$\text{book}_{\text{IOC}}:\ \sigma:\ \big[[p_r,\ q_r - f,\ o_r]\ \text{if } q_r - f > 0\big],\quad \bar\sigma:\ []$$

$$\text{book}_{\text{NON}} = \text{book}_{\text{IOC}} \;+\; \bar\sigma:\ \big[[p_a,\ q_a - f,\ o_a]\ \text{if } q_a - f > 0\big]$$

$$\text{IOC fate } [\text{filled},\ \text{resting},\ \text{hidden},\ \text{evicted},\ \text{cancelled}] = [\,f,\ 0,\ 0,\ 0,\ q_a - f\,]$$

The §5.1 status this signature stands for (finding F2; stated, not asserted as an engine value): **none** `CANCELLED` (unmatched, $f = 0$); **partial** `PARTIALLY_EXECUTED`, remainder `CANCELLED` ($0 < f < q_a$; finding F3); **full** `EXECUTED` ($f = q_a$).

**What is asserted, per valid draw** (each with its expected and observed values recorded in full):

- **(A) acceptance** — order ids 1 and 2 accepted, none rejected, in **both** runs. In regime none this is what separates "cancelled" from "rejected".
- **(Q) counterparty sequence** — the IOC run's fill list, `[passive, aggressor, price_int, qty_int, submission]`, equal to $\text{fills}_{\text{IOC}}$.
- **(M3) trade lists** — the IOC run's fills equal the NON run's, in order, on **every** `Fill` field (finding F5).
- **(R) zero residual** — the IOC's fate equals $[f, 0, 0, 0, q_a - f]$, where filled comes from the engine's trade rows, resting and hidden from the final book and the iceberg channel, evicted from the book diff, and cancelled is the builder's `ioc_remainder` bucket (finding F2); the IOC carries no other cancellation cause; and the IOC's order id appears in **no** post-submission book, from its own submission on.
- **(B) final books** — `[price_int, qty_int, oid]` per side, both runs, equal to $\text{book}_{\text{IOC}}$ and $\text{book}_{\text{NON}}$.
- **(M3-B) book relation** — the IOC run's final book equals the NON run's final book with the NON aggressor's own rows removed, on every column the book exposes (price, quantity, order id, agent, timestamp): the maker survives identically in both.
- **(E) engine reports** — for each run and submission, `[accepted, traded, quantity in ticks]` $= [\text{True}, \text{False}, 0]$ for the maker and $[\text{True}, f > 0, f]$ for the aggressor; on a traded submission, the reported price in ticks within $[p_r - 1, p_r + 1]$ (`ENGINEERING INVARIANT`, finding F7).
- **Battery** — RB and I1–I9 on both jitted records (`expects_trade` $= f > 0$); the battery's own `check_m3_ioc_non_equivalence` on the pair. A draw that either run's I8 marks `invalid` is discarded.

The code below implements (W) as one function of a tape's rows, and the closed form as one function of `draw.params`; section 7 checks that they agree on every draw. Neither reads `draw.expected`, and neither runs the engine.

In [ ]:
def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry.
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


def book_view(entries):
    # {side: sorted [price_int, qty_int, oid]} of the walk's resting entries.
    out = {"BUY": [], "SELL": []}
    for e in entries:
        out[side_name(e["side"])].append([e["price"], e["qty"], e["oid"]])
    return {k: sorted(v) for k, v in out.items()}


def walk(tape):
    # (W): the reference walk over the real rows. The restriction is read only after matching.
    book, fills, cancelled = [], [], {}
    for s, r in enumerate(real_rows(tape)):
        side, p, q, oid, ex = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID]), int(r[C_EXEC])
        left = q
        while left > 0:
            opp = [e for e in book if e["side"] != side and crosses(e["side"], e["price"], p)]
            if not opp:
                break
            best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["entry"]))
            f = min(left, best["qty"])
            fills.append([best["oid"], oid, best["price"], f, s])
            best["qty"] -= f
            left -= f
            if best["qty"] == 0:
                book.remove(best)
        if left > 0:
            if ex == IOC:
                cancelled[oid] = left            # §6.2 item 5, §4.5 IOC row: cancelled, never rests
            else:
                book.append({"side": side, "price": p, "qty": left, "oid": oid, "entry": s})   # §6.2 item 4
    return {"fills": fills, "book_final": book_view(book), "cancelled": cancelled}


def regime_of_tape(tape):
    # The regime, re-derived from the tape's two rows (section 4).
    m, a = real_rows(tape)[:2]
    if not crosses(m[C_SIDE], m[C_PRICE], a[C_PRICE]):
        return "none"
    return "partial" if int(a[C_QTY]) > int(m[C_QTY]) else "full"


def status_of(f, q_agg):
    # The §5.1 status the IOC's fate stands for (stated, not asserted: finding F2).
    if f == 0:
        return "CANCELLED (unmatched)"
    return "EXECUTED" if f == q_agg else "PARTIALLY_EXECUTED, remainder CANCELLED"


def expectation(draw):
    # Everything section 7 compares, from (W) on both tapes' rows.
    rows = real_rows(draw.tape)
    m, a = rows[0], rows[1]
    w_ioc, w_non = walk(draw.tape), walk(counterfactual(draw))
    agg_oid, agg_sub, q_agg = int(a[C_OID]), 1, int(a[C_QTY])
    f = sum(x[3] for x in w_ioc["fills"] if x[4] == agg_sub)
    p_rest = int(m[C_PRICE])
    return {
        "oids": sorted(int(r[C_OID]) for r in rows), "agg_oid": agg_oid, "agg_sub": agg_sub,
        "regime": regime_of_tape(draw.tape), "traded": f,
        "fills": w_ioc["fills"], "fills_non": w_non["fills"],
        "book_ioc": w_ioc["book_final"], "book_non": w_non["book_final"],
        "fate": [f, 0, 0, 0, int(w_ioc["cancelled"].get(agg_oid, 0))],
        "status": status_of(f, q_agg),
        "reports": [[True, False, 0], [True, f > 0, f]],
        "price_bracket": [p_rest - 1, p_rest + 1] if f > 0 else None,
    }


def closed_form(p):
    # The closed form of the formulas above, from draw.params only.
    reg = p["regime"]
    P_r, Q_r, P_a, Q_a = int(p["p_rest_int"]), int(p["q_rest_int"]), int(p["p_agg_int"]), int(p["q_agg_int"])
    o_r, o_a = int(p["rest_oid"]), int(p["agg_oid"])
    f = {"none": 0, "partial": Q_r, "full": Q_a}[reg]
    rs, ag = side_name(p["rest_side"]), side_name(p["agg_side"])
    book_ioc = {rs: ([[P_r, Q_r - f, o_r]] if Q_r - f > 0 else []), ag: []}
    book_non = {rs: list(book_ioc[rs]), ag: ([[P_a, Q_a - f, o_a]] if Q_a - f > 0 else [])}
    fills = [[o_r, o_a, P_r, f, 1]] if f > 0 else []
    return {"fills": fills, "fills_non": fills, "book_ioc": book_ioc, "book_non": book_non,
            "fate": [f, 0, 0, 0, Q_a - f], "status": status_of(f, Q_a), "traded": f, "regime": reg}


def closed_form_agrees(draw):
    try:
        cf, e = closed_form(draw.params), expectation(draw)
    except Exception:  # noqa: BLE001  (a malformed params dict is a disagreement, recorded by the caller)
        return False
    return all(e[k] == cf[k] for k in cf)


def precondition(draw):
    # Section 4's dataset precondition, from both tapes and params. Returns the list of departures.
    bad, t, p = [], np.asarray(draw.tape), draw.params
    cf = counterfactual(draw)
    if cf is None:
        return ["no counterfactual tape"]
    rows = real_rows(t)
    if len(rows) != 2:
        return [f"{len(rows)} real rows (expected 2)"]
    m, a = rows
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids = [int(x[C_OID]) for x in rows]
    if oids != [1, 2] or oids != [int(p.get("rest_oid", -1)), int(p.get("agg_oid", -1))]:
        bad.append(f"order ids {oids} vs params")
    if not int(m[C_TS]) < int(a[C_TS]):
        bad.append(f"tape timestamps not strictly increasing: {[int(m[C_TS]), int(a[C_TS])]}")
    if any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX
           for x in rows):
        bad.append("price or quantity outside §2.3")
    if any(int(x[C_VALIDITY]) != GFS or int(x[C_PEAK]) != 0 or int(x[C_PPD]) != 0 for x in rows):
        bad.append("a row is not GFS, or carries a peak or a price increment")
    if int(m[C_EXEC]) != NON or int(a[C_EXEC]) != IOC:
        bad.append(f"restrictions on the IOC tape: maker {int(m[C_EXEC])}, aggressor {int(a[C_EXEC])}")
    if cf.shape != t.shape or cf.dtype != t.dtype:
        bad.append(f"counterfactual shape/dtype {cf.shape}/{cf.dtype} != {t.shape}/{t.dtype}")
    else:
        agg_pos = int(np.flatnonzero(t[:, C_OID] == oids[1])[0])
        diff = [(int(i), int(j)) for i, j in zip(*np.nonzero(cf != t))]
        if diff != [(agg_pos, C_EXEC)] or int(cf[agg_pos, C_EXEC]) != NON:
            bad.append(f"counterfactual differs from the tape at cells {diff} (expected only the aggressor's "
                       f"restriction, IOC -> NON)")
    rs = int(p.get("rest_side", -1))
    if int(m[C_SIDE]) != rs or int(a[C_SIDE]) != other(rs) or int(p.get("agg_side", -1)) != other(rs):
        bad.append("sides")
    if [int(m[C_PRICE]), int(m[C_QTY]), int(a[C_PRICE]), int(a[C_QTY])] != [
            int(p.get(k, -1)) for k in ("p_rest_int", "q_rest_int", "p_agg_int", "q_agg_int")]:
        bad.append("params p_rest_int / q_rest_int / p_agg_int / q_agg_int != tape")
    if p.get("regime") not in REGIMES or regime_of_tape(t) != p.get("regime"):
        bad.append(f"regime: params {p.get('regime')!r}, tape {regime_of_tape(t)!r}")
    if max_slices(t) != 0 or max_slices(cf) != 0:
        bad.append("an iceberg row")
    return bad


# ---- the expectation on the first seed of each regime -----------------------------------------------------
CF_AGREES = True
for _r in REGIMES:
    if _r not in FIRST_OF:
        continue
    _s = FIRST_OF[_r]
    _d = DRAWN[_s]
    _e = expectation(_d)
    _ok = closed_form_agrees(_d)
    CF_AGREES &= _ok
    print(f"regime {_r}, seed {_s}: traded f = {_e['traded']} (q_agg = {_d.params['q_agg_int']}); "
          f"status: {_e['status']}")
    print(f"    (Q) fills {_e['fills']}   (M3) NON fills {_e['fills_non']}")
    print(f"    (R) IOC fate [filled, resting, hidden, evicted, cancelled] = {_e['fate']}")
    print(f"    (B) IOC book {_e['book_ioc']}   NON book {_e['book_non']}")
    print(f"    (E) reports [accepted, traded, qty_int] {_e['reports']}, price bracket {_e['price_bracket']}")
    print(f"    closed form == walk: {_ok}; precondition departures: {precondition(_d)}")
print(f"closed form agrees with (W) on the first seed of each regime: {CF_AGREES}")

## 6. Run

### Eager and jitted (wired at K3)

> **K2 STUB — section 6 is not wired yet.** **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **one name** from this section. Until it is callable, section 7 records a failing `section 6 wired` check and the notebook ends `NOT CERTIFIED`.

**`run_tape(tape, *, jit) -> RunRecord`**: the battery's record builder on one tape, i.e. `tests.omiecid.reference.omie_invariants.build_record(tape, jit=jit)`. It runs on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), no cancellations (`cancels_before={}`), no gate-closure expiry, and the default capacity bounds (`tape_capacity_bounds(tape)`). `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. Section 7 calls it on **two** tapes per draw, `draw.tape` (the IOC tape) and `draw.aux_tapes["counterfactual"]` (the NON tape), each passed **as is**: nothing in either may be rewritten, and each run starts from its own fresh book.

Section 7 reads these `RunRecord` fields:
- `fills`: every field of each `Fill` (`dataclasses.fields(Fill)`), in particular `passive_oid`, `aggressor_oid`, `price_int`, `qty_int`, `submission`, `row_index`, `sec`, `nsec`;
- `accepted` (its keys and each `AcceptedOrder.submission`) and `rejections` (each `Rejection.oid`);
- `filled`, `still_resting`, `hidden_remaining`, `evicted` and `cancelled_or_expired` (each `Cancellation.cause`, `qty_int`) for the aggressor's order id;
- `post_snapshots[i].resting(side)` and `final_snapshot.resting(side)` (price, qty, oid, agent, sec, nsec);
- `engine_reports`: each `EngineReport`'s `submission`, `accepted`, `traded`, `trade_qty_mw` and `trade_price_eur`;
- `observed_capacity`;
- the whole record, through `run_battery`, `check_m3_ioc_non_equivalence` and `compare_records`.

Section 7 calls `run_tape(..., jit=True)` on both tapes of every draw, and `run_tape(..., jit=False)` on both tapes of the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_tape(tape, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one fresh book of one OPEN contract, with the default
# configs (OMIEProductConfig(), OMIEMarketConfig()), no scheduled cancellations, no
# gate-closure expiry and the default analytic capacity bounds (tape_capacity_bounds(tape)).
# jit=False wraps the whole run in jax.disable_jit() (inside build_record); jit=True runs it
# jitted. Section 7 calls it on draw.tape (the IOC tape) and on
# draw.aux_tapes["counterfactual"] (the NON tape), each passed as is: no column is
# rewritten, and every call starts from its own fresh book.
# ---------------------------------------------------------------------------------------
from omiecid_simulator.omiecid.contract import ContractStatus
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from tests.omiecid.reference.omie_invariants import build_record

MARKET = OMIEMarketConfig()


def run_tape(tape, *, jit):
    """One run of `tape` (a draw's IOC tape or its NON counterfactual, unmodified) on a fresh OPEN book."""
    return build_record(tape, product_config=CONFIG, market_config=MARKET,
                        contract_status=ContractStatus.OPEN, cancels_before={},
                        expire_at_gct=False, capacity_bounds=None, jit=jit)


print(f"run_tape: build_record on one OPEN contract, {type(CONFIG).__name__}() + {type(MARKET).__name__}(), "
      f"no cancels, no GCT expiry, default capacity bounds; called on the IOC tape and its NON counterfactual")

## 7. Certification

The oracle comparison comes first, by order id and in integer ticks; then the metamorphic relation M3 between each draw's IOC run and its NON run. Then come the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw**, each with its expected and observed value **in full**, so an auditor can check every integer from the results JSON alone: (A) accepted and rejected order ids of both runs; (Q) the IOC run's counterparty sequence; (M3) the NON run's full trade rows (expected) beside the IOC run's (observed); (R) the IOC's fate, its other cancellation causes, and the post-submission books that hold its id; (B) both final books; (M3-B) the NON book less the NON aggressor (expected) beside the IOC book (observed), every exposed column; (E) both runs' engine reports in ticks, beside the raw reported price. The draw's parameters are recorded once, with its tape (`record_case`), and section 5's closed form is not repeated, only its agreement.

**Aggregate assertions:** the section-6 wiring; the closed form against (W), on the first seeds and on every valid draw; the dataset precondition (section 4), listing every departing seed and its departures; **coverage** — all six classes regime × resting side among the valid draws; the eager comparisons, one per class and tape, a missing one recorded as a failure.

The battery runs on both jitted records of every draw with `expects_trade` $= f > 0$. The IOC run's battery is folded into the invariant table below; the NON counterfactual's is recorded as one assertion per draw (its failing checks, expected none), so a draw is counted once. A draw that either run's I8 marks `invalid` is discarded and replaced by the next seed. **Reported, not asserted:** the regime counts of the valid draws, and the analytic hop-bound peak.

In [ ]:
import dataclasses
import math
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import (
    Fill, check_m3_ioc_non_equivalence, compare_records, run_battery,
)

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Immediate-or-cancel",
    known_result="IOC and NON with identical price and quantity produce identical trade lists, field for field "
                 "(M3): [[o_r, o_a, p_r, f, 1]] if f > 0, with f = 0 (none), q_r (partial), q_a (full); the IOC's "
                 "own id never rests and its fate is [filled, resting, hidden, evicted, cancelled] = "
                 "[f, 0, 0, 0, q_a - f] (status CANCELLED when unmatched, remainder CANCELLED when partial, "
                 "EXECUTED when full; status itself unobserved); the NON's leftover q_a - f rests at p_a; the "
                 "maker keeps q_r - f in both runs; every order accepted",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS, constructions=(CONSTRUCTION,),
    requested_draws=DRAWS, runtime_forecast_s=120.0, started_at=T0,
    downstream="every scenario and component that submits IOC orders: C10 (FOK shares the 'never enters the "
               "book' branch of §4.5 and §5.1), C15 (the non_ioc fuzz regimes mix IOC and NON); "
               "invariant I2's ioc_remainder bucket; Track B, whose RL agent submits IOC orders, so its fills "
               "and its never-resting assumption rest on this rule",
)

R_ACC = "§5.1 SUBMITTED -> ACCEPTED (an unmatched IOC is cancelled, not rejected)"
R_SEQ = "§6.1; §6.2 items 1-3 (best counterpart, crossing, maker price)"
R_M3 = "M3; §6.2 items 1-5 (matching does not depend on the restriction); §4.5 IOC and NON rows"
R_RES = "§4.5 IOC row (no book entry); §6.2 item 5; §5.1 CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; " \
        "halt/hibernation)"
R_BOOK = "§4.5 IOC and NON rows; §6.2 items 4-5"
R_ENG = "ENGINEERING INVARIANT (report price); §5.1 accepted; §6.2 (traded quantity)"
R_BAT = "battery RB, I1-I9 on the NON counterfactual"
R_DATA = "dataset precondition (section 4); §2.3"
FILL_FIELDS = tuple(f.name for f in dataclasses.fields(Fill))


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_ACC)
        return None


def to_ticks(x, origin, tick):
    # A float report figure in ticks (round to nearest); None when it is not a finite number.
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return int(round((v - float(origin)) / float(tick))) if math.isfinite(v) else None


def full_fills(rec):
    # Every Fill field of every trade row, in (submission, row_index) order.
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    return [[int(getattr(f, k)) for k in FILL_FIELDS] for f in fills]


def short_fills(rec):
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    return [[int(f.passive_oid), int(f.aggressor_oid), int(f.price_int), int(f.qty_int), int(f.submission)]
            for f in fills]


def book3(snap):
    return {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in snap.resting(sd)) for sd in (BUY, SELL)}


def book_full(snap, drop_oid=None):
    # Every column the book exposes: (price, qty, oid, agent, sec, nsec), optionally without one order id.
    return {side_name(sd): sorted([int(v) for v in x] for x in snap.resting(sd) if int(x[2]) != drop_oid)
            for sd in (BUY, SELL)}


def fate(rec, oid):
    canc = rec.cancelled_or_expired.get(oid, ())
    return ([int(rec.filled.get(oid, 0)), int(rec.still_resting.get(oid, 0)), int(rec.hidden_remaining.get(oid, 0)),
             int(rec.evicted.get(oid, 0)), sum(int(c.qty_int) for c in canc if c.cause == "ioc_remainder")],
            sorted({str(c.cause) for c in canc if c.cause != "ioc_remainder"}))


def holding(rec, oid):
    # The post-submission snapshots, from the order's own submission on, whose book holds oid.
    a = rec.accepted.get(oid)
    start = int(a.submission) if a is not None else 0
    return [i for i in range(start, len(rec.post_snapshots))
            if any(int(r[2]) == oid for sd in (BUY, SELL) for r in rec.post_snapshots[i].resting(sd))]


def reports(rec):
    # [[accepted, traded, qty_int]] per submission, and the reported price in ticks per submission.
    ers = sorted(rec.engine_reports, key=lambda r: int(r.submission))
    rows = [[bool(r.accepted), bool(r.traded), to_ticks(r.trade_qty_mw, 0.0, CONFIG.qty_tick)] for r in ers]
    prices = [to_ticks(r.trade_price_eur, CONFIG.price_min, CONFIG.price_tick) for r in ers]
    raw = [float(r.trade_price_eur) for r in ers]
    return rows, prices, raw


def check_draw(seed, rec, rec_n, e):
    # (A), (Q), (M3), (R), (B), (M3-B), (E) of section 5, on the jitted pair. Returns True when all passed.
    a = e["agg_oid"]
    lbl = f"seed {seed} ({e['regime']})"
    ok = cert.check_equal(
        f"{lbl}: (A) [accepted oids, rejected oids], IOC run and NON run",
        {"IOC": [sorted(int(o) for o in rec.accepted), sorted(int(r.oid) for r in rec.rejections)],
         "NON": [sorted(int(o) for o in rec_n.accepted), sorted(int(r.oid) for r in rec_n.rejections)]},
        {"IOC": [e["oids"], []], "NON": [e["oids"], []]}, seed=seed, unit="other", rule=R_ACC)
    ok &= cert.check_equal(f"{lbl}: (Q) IOC counterparty sequence [passive, aggressor, price_int, qty_int, "
                           f"submission]", short_fills(rec), e["fills"], seed=seed, unit="other", rule=R_SEQ)
    ok &= cert.check_equal(f"{lbl}: (M3) IOC trade rows == NON trade rows, every Fill field {list(FILL_FIELDS)}",
                           full_fills(rec), full_fills(rec_n), seed=seed, unit="other", rule=R_M3)
    fa, other_causes = fate(rec, a)
    ok &= cert.check_equal(
        f"{lbl}: (R) zero residual: IOC oid {a} fate [filled, resting, hidden, evicted, ioc_remainder], other "
        f"cancellation causes, post-submission books holding it ({e['status']})",
        {"fate": fa, "other_causes": other_causes, "post_snapshots_holding": holding(rec, a)},
        {"fate": e["fate"], "other_causes": [], "post_snapshots_holding": []}, seed=seed, unit="other", rule=R_RES)
    ok &= cert.check_equal(f"{lbl}: (B) final books [price_int, qty_int, oid], IOC run and NON run",
                           {"IOC": book3(rec.final_snapshot), "NON": book3(rec_n.final_snapshot)},
                           {"IOC": e["book_ioc"], "NON": e["book_non"]}, seed=seed, unit="other", rule=R_BOOK)
    ok &= cert.check_equal(f"{lbl}: (M3-B) IOC final book == NON final book less oid {a} "
                           f"[price, qty, oid, agent, sec, nsec]", book_full(rec.final_snapshot),
                           book_full(rec_n.final_snapshot, drop_oid=a), seed=seed, unit="other", rule=R_M3)
    rows_i, prices_i, raw_i = reports(rec)
    rows_n, prices_n, raw_n = reports(rec_n)
    br = e["price_bracket"]
    price_ok = br is None or all(p is not None and br[0] <= p <= br[1] for p in (prices_i[1:2] + prices_n[1:2]))
    rep_ok = rows_i == e["reports"] and rows_n == e["reports"] and len(prices_i) == 2 and price_ok
    ok &= cert.check(
        f"{lbl}: (E) engine reports [accepted, traded, qty_int] per submission, both runs; aggressor's reported "
        f"price_int within [p_rest - 1, p_rest + 1] when traded (ENGINEERING INVARIANT)", rep_ok, seed=seed,
        expected={"IOC": e["reports"], "NON": e["reports"], "agg_price_int_in": br},
        observed={"IOC": rows_i, "NON": rows_n,
                  "agg_price_int": {"IOC": prices_i[1:2], "NON": prices_n[1:2]},
                  "agg_price_eur": {"IOC": raw_i[1:2], "NON": raw_n[1:2]}},
        unit="other", rule=R_ENG)
    m3 = check_m3_ioc_non_equivalence(rec, rec_n, oid=a)
    ok &= cert.check(f"{lbl}: battery M3 (check_m3_ioc_non_equivalence)", m3.status == "pass", seed=seed,
                     expected="pass", observed=m3.status,
                     detail="; ".join(v.message for v in m3.violations), unit="other", rule=R_M3)
    return bool(ok)


EAGER_DONE = {}


def eager_compare(seed, which, tape, rec, tags):
    # Run `tape` eager and compare it with its jitted record (one eager_jit assertion).
    rec_e = guarded(f"seed {seed} {which} (eager)", seed, run_tape, tape, jit=False)
    if rec is None or rec_e is None:
        diff = ["jitted run missing" if rec is None else "eager run missing"]
    else:
        diff = list(compare_records(rec_e, rec))
    for t in tags:
        EAGER_DONE.setdefault((t, which), seed)
    cert.check(f"eager == jitted (seed {seed}, {which} tape)", diff == [], kind="eager_jit", seed=seed,
               expected=True, observed=diff == [], unit="bool",
               detail=", ".join(diff) if diff else "identical")
    return diff == []


def margin_breaches(tape, bounds):
    # The capacity-validity precondition (five ceilings), checked from the tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = [f"{k} bound {bounds.get(k)} > {m}" for k, m in lim.items() if int(bounds.get(k, 0)) > m]
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


cert.check("section 5: closed form agrees with the walk (W) on the first seed of each regime", CF_AGREES,
           expected=True, observed=CF_AGREES, unit="bool", rule=R_SEQ)
wired = callable(globals().get("run_tape"))
cert.check("section 6 wired (run_tape defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_ACC)

CLASSES = [f"{r}, resting {s}" for r in REGIMES for s in ("BUY", "SELL")]
PER_RUN, DATA_BAD, CLOSED_BAD, VALID_SEEDS = [], {}, [], []
COVER = Counter()
PEAK = {"hop_bound": 0, "max_slices": 0}


def run_draws():
    valid, seed = 0, SEED0
    while valid < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
        draw = DRAWN.get(seed) or prepare(seed)
        cf = counterfactual(draw)
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if cf is not None:
            margin += [f"counterfactual: {x}" for x in margin_breaches(cf, synth.tape_capacity_bounds(cf))]
        if margin:
            cert.discard(seed, "pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        bad = precondition(draw)
        if bad:
            DATA_BAD[seed] = bad
            cert.check(f"seed {seed}: draw is as section 4 describes", False, seed=seed, expected=[],
                       observed=bad, unit="other", rule=R_DATA)
            seed += 1
            continue
        e = expectation(draw)
        rec = guarded(f"seed {seed} IOC (jit)", seed, run_tape, draw.tape, jit=True)
        rec_n = guarded(f"seed {seed} NON (jit)", seed, run_tape, cf, jit=True)
        rep = run_battery(rec, expects_trade=e["traded"] > 0) if rec is not None else None
        rep_n = run_battery(rec_n, expects_trade=e["traded"] > 0) if rec_n is not None else None
        if rep_n is not None and not rep_n.valid:
            cert.discard(seed, "NON counterfactual: I8 invalid: generated draw exceeds the capacity envelope")
            seed += 1
            continue
        if rep is not None and not cert.add_battery(rep, seed=seed):
            seed += 1   # I8 invalid on the IOC run: discarded, re-drawn with the next seed
            continue
        valid += 1
        VALID_SEEDS.append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": CONSTRUCTION})
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound),
                                int(synth.tape_capacity_bounds(cf)["hops"]))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape), max_slices(cf))
        tag = f"{e['regime']}, resting {side_name(draw.params['rest_side'])}"
        COVER[tag] += 1
        if not closed_form_agrees(draw):
            CLOSED_BAD.append(seed)
        for r in (rec, rec_n):
            if r is not None:
                cert.record_capacity({**r.observed_capacity, "hops": None})   # hops: unobserved by the engine
        if rep_n is not None:
            bad_n = {r.check_id: r.status for r in rep_n.results if r.status not in ("pass", "not_applicable")}
            cert.check(f"seed {seed} ({e['regime']}): NON counterfactual battery RB, I1-I9 (failing checks)",
                       not bad_n, seed=seed, expected={}, observed=bad_n, unit="other", rule=R_BAT,
                       detail="; ".join(v.message for r in rep_n.results for v in r.violations)[:2000])
        if rec is not None and rec_n is not None:
            PER_RUN.append({"seed": seed, "regime": e["regime"], "ok": check_draw(seed, rec, rec_n, e)})
        if valid <= EAGER_DRAWS or any((tag, w) not in EAGER_DONE for w in ("IOC", "NON")):
            eager_compare(seed, "IOC", draw.tape, rec, [tag])
            eager_compare(seed, "NON", cf, rec_n, [tag])
        seed += 1
    return valid


VALID = run_draws() if wired else 0

# ---- aggregate assertions (every list in full) ----------------------------------------------------------
cert.check("section 5: closed form agrees with the walk (W) on every valid draw", wired and not CLOSED_BAD,
           expected=[], observed=CLOSED_BAD, unit="other", rule=R_SEQ)
cert.check("dataset: every draw is as section 4 describes (precondition)", wired and not DATA_BAD,
           expected={}, observed={str(k): v for k, v in DATA_BAD.items()}, unit="other",
           detail=f"{len(DATA_BAD)} draws depart" if DATA_BAD else f"{VALID} valid draws", rule=R_DATA)
_got = {c: COVER[c] for c in CLASSES}
_missing = [c for c, v in _got.items() if v == 0]
cert.check("dataset: the valid draws cover every regime x resting side (none, partial, full; BUY, SELL)",
           wired and not _missing, expected={c: ">= 1" for c in CLASSES}, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)

# ---- eager: every class, on both tapes, must have had one comparison (absence = failure) ---------------------
for _c in CLASSES:
    for _w in ("IOC", "NON"):
        if (_c, _w) not in EAGER_DONE:
            cert.check(f"eager == jitted for class '{_c}', {_w} tape", False, kind="eager_jit", expected=True,
                       observed=None, unit="bool", detail=f"no eager comparison was run for class '{_c}', {_w} tape")

# ---- reports (not asserted) ----------------------------------------------------------------------------------
_fail_runs = [r for r in PER_RUN if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_RUN)} draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"seed {r['seed']} ({r['regime']})" for r in _fail_runs) if _fail_runs else ".")))
print(f"valid draws: {VALID}; per class: {dict(COVER)}")
print(f"eager comparisons run for: {sorted(EAGER_DONE)}")
print(f"analytic peaks over valid draws (both tapes): hop bound {PEAK['hop_bound']}, max ceil(qty/peak) "
      f"{PEAK['max_slices']}")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))